In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler import generate_preset_pass_manager
import json

print("Imports loaded successfully — you're ready to go.")

Imports loaded successfully — you're ready to go.


In [2]:
with open("api.json", "r") as f:
    credentials = json.load(f)

api_key = credentials["apikey"]
crn = credentials["crn"]

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    instance=crn,
    token=api_key,
    set_as_default=True,
    overwrite=True
)

# You only need to do this once per environment.

In [3]:
service = QiskitRuntimeService()
real_backend = service.least_busy(operational=True, simulator=False)
sampler = SamplerV2(real_backend)

In [4]:
# Simple Bell-state circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Transpile for the selected hardware
pm = generate_preset_pass_manager(backend=real_backend, optimization_level=1)
qc = pm.run(qc)

# Submit job
job = sampler.run([qc], shots=100)
print("Backend:", real_backend.name)
print("Job ID:", job.job_id())

# Retrieve results
print("Counts:", job.result()[0].data.meas.get_counts())

Backend: ibm_quebec
Job ID: dbnjnfs4148s73915bfg
Counts: {'00': 45, '10': 1, '11': 54}
